# Instruction the lab-work #4
`TODO`: make the final .ipynb file in format `"{second_name}_{first_name}_{group_id}__lab_3.ipynb"`

First name     - Volodymyr

Second name    - Donets

Group          - KU-31

# Task description

## Overall description

In this lab work you'll try to process sequential data by 2 types of neural networks. The first one implement constant input, the second one allow non-constant input.


## *Instructions for the work*

1. Select a dataset you interested in (do its preparation, for the huge datasets use generators)
2. Carefully look at examples.
3. Develop the model with fixed input size (create it's architecture & train)
4. Develop the model with flexible input size (create it's architecture & train)
5. Make conclusions on the model performance.

## Examples

1. [Source example for the labs code](https://www.digitalocean.com/community/tutorials/audio-classification-with-deep-learning)

## Important

1. Do NOT solve the problem in a single cell, such work will be worth 0. If you struggle with Jupyter Notebook just ask for help teacher or your friends. Also, you can solve the lab work in .py files, but provide your report with in .pdf file in the similar format.
2. Do NOT try to steal others works. This year I'll try to consider this. And identical works will be worth 0 too.
3. You can (and probably should) solve the task of the lab work in the separate .ipynb file. Just take this file, clear my solution and replace it with your own.
4. Do NOT forget to leave CONCLUSIONS.

## How to get max num of points without personal work presentation.

As I told you, you can get the maximum mark for all lab works without personal presentation in case of using something different and additional depth of work.
So for this work you can:

1. Run multiple types of architectures (like fully connected and conv for processing sequential data with constant input).
2. Make a few experiments with data preparation like using various data compression techniques (like PCA).
3. Work with complex data (like sound, text, any other sequential data like time series forecasting)

# Import & install dependencies

In [ ]:
%pip install tensorflow
%pip install tensorflow-io
%pip install kaggle
%pip install librosa          # this lib required for tensorflow_io on Windows

In [15]:
import os

import numpy as np
import tensorflow as tf 
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Input
import librosa

from matplotlib import pyplot as plt

# Load the data

In [ ]:
# Load the data
!kaggle datasets download kenjee/z-by-hp-unlocked-challenge-3-signal-processing

In [3]:
CAPUCHIN_FILE = os.path.join("src", "Parsed_Capuchinbird_Clips", "XC3776-3.wav")
NOT_CAPUCHIN_FILE = os.path.join("src", "Parsed_Not_Capuchinbird_Clips", "afternoon-birds-song-in-forest-0.wav")

In [4]:
def load_wav_16k_mono(filename):
    # librosa.load automatically resamples to the target sr (16000) 
    # and converts to mono by default. It returns a numpy array.
    # We decode the bytes filename to a string if it's passed as a tensor.
    if isinstance(filename, tf.Tensor):
        filename = filename.numpy().decode('utf-8')
        
    wav, sample_rate = librosa.load(filename, sr=16000, mono=True)
    
    # Convert back to a TensorFlow tensor
    return tf.convert_to_tensor(wav, dtype=tf.float32)


# --- If you are using this inside a tf.data.Dataset.map() ---
# You will need to wrap it in tf.numpy_function because librosa runs outside the TF graph:
def tf_load_wav_16k_mono(filename):
    wav = tf.numpy_function(load_wav_16k_mono, [filename], tf.float32)
    # Ensure the tensor has a shape (numpy_function forgets shapes)
    wav.set_shape([None]) 
    return wav

In [5]:
# Defining the positive and negative paths
POS = os.path.join('src', 'Parsed_Capuchinbird_Clips/*.wav')
NEG = os.path.join('src', 'Parsed_Not_Capuchinbird_Clips/*.wav')

# Creating the Datasets
pos = tf.data.Dataset.list_files(POS)
neg = tf.data.Dataset.list_files(NEG)

# Adding labels
positives = tf.data.Dataset.zip((pos, tf.data.Dataset.from_tensor_slices(tf.ones(len(pos)))))
negatives = tf.data.Dataset.zip((neg, tf.data.Dataset.from_tensor_slices(tf.zeros(len(neg)))))
data = positives.concatenate(negatives)

In [6]:
lengths = []

# Load all audio files and store their lengths
for file in os.listdir(os.path.join('src', 'Parsed_Capuchinbird_Clips')):
    tensor_wave = load_wav_16k_mono(os.path.join('src', 'Parsed_Capuchinbird_Clips', file))
    lengths.append(len(tensor_wave))

# Convert lengths to TensorFlow tensors
lengths_tensor = tf.constant(lengths, dtype=tf.int32)

# Calculate statistics
min_length = tf.reduce_min(lengths_tensor)
mean_length = tf.reduce_mean(lengths_tensor)
max_length = tf.reduce_max(lengths_tensor)

# Display the results
print("Minimum waveform length:", min_length)
print("Mean waveform length:", mean_length)
print("Maximum waveform length:", max_length)

Minimum waveform length: tf.Tensor(32000, shape=(), dtype=int32)
Mean waveform length: tf.Tensor(54156, shape=(), dtype=int32)
Maximum waveform length: tf.Tensor(80000, shape=(), dtype=int32)


In [8]:
def preprocess(file_path, label): 
    # 1. Decode the byte string to a regular string if necessary
    if isinstance(file_path, bytes):
        file_path = file_path.decode('utf-8')
    elif isinstance(file_path, tf.Tensor):
        # In case you map this function later, handle the Tensor format
        file_path = file_path.numpy().decode('utf-8')

    # 2. Load the audio file (using your librosa-backed function)
    wav = load_wav_16k_mono(file_path)
    
    # 3. Slice the audio to 48000 samples (exactly 3 seconds)
    wav = wav[:48000]  
    
    # 4. Pad with zeros if the audio is shorter than 3 seconds
    zero_padding = tf.zeros([48000] - tf.shape(wav), dtype=tf.float32)
    wav = tf.concat([zero_padding, wav], axis=0)

    # 5. Convert waveform to spectrogram
    spectrogram = tf.signal.stft(wav, frame_length=320, frame_step=32)
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.expand_dims(spectrogram, axis=2)  # Add channel dimension
    
    return spectrogram, label


# Shuffle and get a sample
filepath, label = positives.shuffle(buffer_size=10000).as_numpy_iterator().next()

# Preprocess the sample
spectrogram, label = preprocess(filepath, label)

In [9]:
print("Spectrogram shape:", spectrogram.shape)
print("Label:", label)

Spectrogram shape: (1491, 257, 1)
Label: 1.0


In [12]:
# 1. The pure Python function that librosa can understand
def load_audio_librosa(file_path):
    # tf.numpy_function passes strings as bytes, so we decode here
    file_path_str = file_path.decode('utf-8')
    wav, _ = librosa.load(file_path_str, sr=16000, mono=True)
    # Ensure it returns a float32 numpy array for TensorFlow
    return wav.astype(np.float32) 

# 2. The TensorFlow function for dataset.map()
def preprocess(file_path, label): 
    # Wrap the librosa call in tf.numpy_function
    wav = tf.numpy_function(load_audio_librosa, [file_path], tf.float32)
    
    # CRITICAL: tf.numpy_function forgets tensor shapes. 
    # We must tell TF it's a 1D array of unknown length so downstream math works.
    wav.set_shape([None])
    
    # Slicing and padding (TF math)
    wav = wav[:48000]  
    zero_padding = tf.zeros([48000] - tf.shape(wav), dtype=tf.float32)
    wav = tf.concat([zero_padding, wav], axis=0)

    # Convert waveform to spectrogram (TF math)
    spectrogram = tf.signal.stft(wav, frame_length=320, frame_step=32)
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.expand_dims(spectrogram, axis=2)  
    
    return spectrogram, label

In [13]:
# Creating a Tensorflow Data Pipeline
data = data.map(preprocess)
data = data.cache()
data = data.shuffle(buffer_size=1000)
data = data.batch(16)
data = data.prefetch(8)

# Split into Training and Testing Partitions
train = data.take(36)
test = data.skip(36).take(15)

In [16]:
model = Sequential()
model.add(Input(shape=(1491, 257,1)))
model.add(Conv2D(16, (3,3), activation='relu'))
model.add(Conv2D(16, (3,3), activation='relu'))
model.add(Flatten())
# model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 1489, 255, 16)  │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 1487, 253, 16)  │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6019376)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │     6,019,377 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,021,857 (22.97 MB)

 Trainable params: 6,021,857 (22.97 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Compiling and fitting the model
model.compile('Adam', loss='BinaryCrossentropy', metrics=[tf.keras.metrics.Recall(),tf.keras.metrics.Precision()])

model.fit(train, epochs=4, validation_data=test)

Epoch 1/4
36/36 ━━━━━━━━━━━━━━━━━━━━ 27s 540ms/step - loss: 0.9799 - precision: 0.8684 - recall: 0.8627 - val_loss: 0.0312 - val_precision: 1.0000 - val_recall: 0.9375
Epoch 2/4
36/36 ━━━━━━━━━━━━━━━━━━━━ 20s 549ms/step - loss: 0.0166 - precision: 1.0000 - recall: 0.9932 - val_loss: 0.0041 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 3/4
36/36 ━━━━━━━━━━━━━━━━━━━━ 19s 533ms/step - loss: 0.0526 - precision: 0.9933 - recall: 0.9868 - val_loss: 0.0091 - val_precision: 1.0000 - val_recall: 0.9846
Epoch 4/4
36/36 ━━━━━━━━━━━━━━━━━━━━ 19s 531ms/step - loss: 0.0078 - precision: 1.0000 - recall: 0.9939 - val_loss: 7.5788e-04 - val_precision: 1.0000 - val_recall: 1.0000
